# Bài 12 · Pooling & kiến trúc CNN kinh điển: tự tính LeNet, AlexNet, VGG, rồi chạm bức tường độ sâu

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/12-kien-truc-cnn.ipynb)

Notebook đi kèm [Bài 12 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/kien-truc-cnn/) (demo tương tác nằm ở trang bài học).
Một CNN hoàn chỉnh xếp xen kẽ tích chập, ReLU và pooling để thu nhỏ ảnh, rồi kết thúc bằng lớp phân loại.
Ở đây ta tự viết bằng numpy từng mảnh của nó, kiểm tra từng công thức bằng cách tính trực tiếp, huấn luyện một LeNet
trên MNIST thật, rồi tính lại từng lớp của ba kiến trúc kinh điển và so với số đã công bố.

**Bạn sẽ làm:**
1. Viết max pooling và average pooling, và xem chúng phản ứng thế nào khi dời ảnh 1 hoặc 2 điểm ảnh.
2. Kiểm tra công thức kích thước đầu ra và công thức vùng tiếp nhận bằng cách đếm trực tiếp.
3. Huấn luyện một LeNet trên MNIST và đo xem dời ảnh 1 điểm ảnh làm bao nhiêu dự đoán thay đổi.
4. Đếm tham số và phép nhân–cộng của LeNet‑5, AlexNet và VGG‑16 từng lớp một, khớp với số trong các bài báo.
5. Đọc kết quả của mạng “trơn” 8–56 lớp và thấy mạng sâu hơn lại học kém hơn: bức tường độ sâu.

**Cần biết trước:** [Bài 11 · Mạng tích chập (CNN)](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/cnn/) (tích chập,
bộ lọc dùng chung), [Bài 06 · Lan truyền ngược](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/)
(dùng ở mục 6) và [Bài 10 · BatchNorm, LayerNorm & RMSNorm](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/chuan-hoa/)
(các mạng sâu ở mục 10 có BatchNorm). **Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong demo. Riêng kết quả huấn luyện
3 seed của 6 mạng LeNet (5 lượt) và của 45 mạng sâu được nạp từ đúng tệp mà demo dùng, vì chạy lại mất hàng chục
phút; notebook tự huấn luyện một LeNet ngắn hơn để thấy cơ chế.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import gzip
import hashlib
import json
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter
from numpy.lib.stride_tricks import sliding_window_view

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
F32 = np.float32


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    s = f"{x:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def vn_axes(fig, which="xy"):
    """Đổi nhãn các trục số của hình sang kiểu Việt Nam; trục log lấy vạch 1–2–5. Gọi ngay trước plt.show().
    which="x" khi trục y mang nhãn chữ (tên mạng…), để giữ nguyên các nhãn đó."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.axis_name not in which:
                continue
            if axis.get_scale() == "log":
                lo, hi = sorted(lim)
                ticks = [m * 10.0**k for k in range(-6, 4) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
                axis.set_major_locator(FixedLocator(ticks))
                axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(vn_tick))


print(vn(13007, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.993527))

## 2 · Dữ liệu

MNIST (LeCun và cộng sự, 1998): 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, chữ số viết tay xám 28×28.
Ô dưới tải 4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu. Ảnh được giữ ở dạng
(số ảnh, cao, rộng, kênh) = (N, 28, 28, 1), giá trị trong [0, 1], kiểu float32 như các script của demo.

Ô cũng nạp hai tệp kết quả mà demo dùng (`fetch_json`, có kiểm tra MD5): 6 cách thu nhỏ của LeNet × 3 seed, và
các mạng “trơn” 8–56 lớp × 3 tốc độ học × 3 seed.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Kết quả của những lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
lenet_runs = fetch_json("mnist-lenet.json", "bf1680c82271d41115500fb24e4e6f8f")  # 6 cách thu nhỏ × 3 seed
depth_runs = fetch_json("mnist-depth.json", "8156284d0611a3445408647bbae5d813")  # mạng trơn 8–56 lớp

In [ ]:
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xtr = (train_img / 255.0).astype(F32)[..., None]  # (60.000, 28, 28, 1)
Xte = (test_img / 255.0).astype(F32)[..., None]
ytr, yte = train_lab.astype(np.int64), test_lab.astype(np.int64)
print("huấn luyện:", Xtr.shape, " kiểm tra:", Xte.shape, " kiểu:", Xtr.dtype)
assert (len(ytr), len(yte)) == (lenet_runs["nTrain"], lenet_runs["nTest"])
n_deep = sum(len(r["seeds"]) for r in depth_runs["runs"])
print(f"tệp LeNet: {len(lenet_runs['runs'])} mạng × {len(lenet_runs['seeds'])} seed · tệp độ sâu: {n_deep} mạng sâu")

Mỗi chữ số một ảnh, lấy từ tập kiểm tra:

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for d, ax in enumerate(axes):
    ax.imshow(Xte[np.flatnonzero(yte == d)[0], :, :, 0], cmap="gray_r")
    ax.set_title(str(d), fontsize=11)
    ax.axis("off")
plt.show()

## 3 · Max pooling thu nhỏ bản đồ đặc trưng

Pooling chia bản đồ thành các ô $k \times k$ và thay mỗi ô bằng một số: **max pooling** giữ giá trị lớn nhất,
**average pooling** lấy trung bình. Với $k = 2$ và bước (stride) $s = 2$, mỗi cạnh giảm một nửa, số giá trị giảm
4 lần. Pooling không có trọng số nào để học.

Bắt đầu với một tín hiệu một chiều và một vòng lặp. Hàm `vec()` in một dãy số theo kiểu Việt Nam.

In [ ]:
def vec(a):
    return "[" + "  ".join(vn_tick(v) for v in a) + "]"


def pool1d_loop(x, k=2, s=2, kind="max"):
    out = []
    for start in range(0, len(x) - k + 1, s):  # các cửa sổ x[start : start + k]
        window = x[start : start + k]
        out.append(max(window) if kind == "max" else sum(window) / k)
    return np.array(out)


x1 = np.array([0, 0, 1, 1, 0, 0, 1, 1, 0, 0], float)
print("x                 =", vec(x1))
print("max pooling 2     =", vec(pool1d_loop(x1)))
print("average pooling 2 =", vec(pool1d_loop(x1, kind="avg")))

10 giá trị thành 5: mỗi cặp $(x_1, x_2), (x_3, x_4), \dots$ còn một số.

### 🤔 Dự đoán trước
Dời $x$ sang phải **1 ô** (ô trống điền 0). Max pooling cho ra gì: vẫn là $[0\ 1\ 0\ 1\ 0]$, hay bản dời của nó,
hay một thứ khác hẳn? Còn khi dời **2 ô**?

In [ ]:
def shift1d(x, d):
    """Dời tín hiệu sang phải d ô, điền 0 vào chỗ trống."""
    return np.concatenate([np.zeros(d), x[: len(x) - d]])


for d in (0, 1, 2):
    xs = shift1d(x1, d)
    print(f"dời {d} ô: x = {vec(xs)}  →  max {vec(pool1d_loop(xs))}   trung bình {vec(pool1d_loop(xs, kind='avg'))}")
same2 = np.array_equal(pool1d_loop(shift1d(x1, 2)), shift1d(pool1d_loop(x1), 1))
print("dời 2 ô rồi pooling = pooling rồi dời 1 ô:", same2)
assert same2

Dời 1 ô, max pooling ra $[0\ 1\ 1\ 1\ 1]$: hai đỉnh tách rời biến thành một dải liền. Đó không phải đầu ra cũ, cũng
không phải bản dời của nó. Average pooling cho $[0\ 0{,}5\ 0{,}5\ 0{,}5\ 0{,}5]$, cũng mất hẳn hình dạng. Chỉ khi dời
đúng **một bội của bước** (2 ô) thì đầu ra mới dời theo, 1 ô.

Zhang (2019) giải thích điều này bằng cách tách max pooling thành hai bước: lấy max trên **mọi** cửa sổ (bước 1, nên
dời đầu vào thì đầu ra dời theo), rồi **lấy mẫu thưa**, giữ một ô trong hai. Bước thứ hai làm mất tính chất đó, vì nó bỏ
qua định lý lấy mẫu (Nyquist): tín hiệu đổi nhanh hơn tốc độ lấy mẫu thì bị “gập” thành một tín hiệu khác (aliasing).
Cách sửa kinh điển trong xử lý ảnh là làm mờ trước khi lấy mẫu thưa; Zhang gọi phiên bản cho max pooling là
**MaxBlurPool**: max 2×2 bước 1, làm mờ bằng bộ lọc $[1\ 2\ 1]/4$ theo mỗi chiều, rồi mới lấy bước 2.

Trên ảnh, ta viết pooling 2×2 bằng cách lấy 4 góc của mọi ô 2×2 (bốn mảng `x[:, i::2, j::2]`). Cạnh lẻ thì
hàng/cột cuối bị bỏ.

In [ ]:
QUAD = [(0, 0), (0, 1), (1, 0), (1, 1)]  # 4 vị trí trong một ô 2×2


def pool2(x, kind="max"):
    """Pooling 2×2 bước 2 trên trục 1 và 2 của x (N, H, W) hoặc (N, H, W, C)."""
    H, W = x.shape[1] // 2 * 2, x.shape[2] // 2 * 2  # cạnh lẻ: bỏ hàng/cột cuối
    q = [x[:, i:H:2, j:W:2] for i, j in QUAD]
    if kind == "avg":
        return (q[0] + q[1] + q[2] + q[3]) / 4
    return np.maximum(np.maximum(q[0], q[1]), np.maximum(q[2], q[3]))


def subsample2(x):
    """Chỉ lấy mẫu thưa (một ô trong hai mỗi chiều), như tích chập bước 2 không có bộ lọc."""
    return x[:, ::2, ::2]


def maxblurpool2(x):
    """Zhang (2019): max 2×2 bước 1, làm mờ 3×3 [1 2 1]/4 ⊗ [1 2 1]/4, rồi lấy bước 2 (như convnet.MaxBlurPool)."""
    H, W = x.shape[1:3]
    rest = [(0, 0)] * (x.ndim - 3)
    xp = np.pad(x, [(0, 0), (0, 1), (0, 1), *rest], mode="edge")  # giữ cỡ H × W sau max bước 1
    m = np.maximum(np.maximum(xp[:, :H, :W], xp[:, 1:, :W]), np.maximum(xp[:, :H, 1:], xp[:, 1:, 1:]))
    mp = np.pad(m, [(0, 0), (1, 1), (1, 1), *rest], mode="reflect")
    w = np.array([1, 2, 1]) / 4
    out = sum(w[i] * w[j] * mp[:, i : i + H, j : j + W] for i in range(3) for j in range(3))
    return out[:, ::2, ::2]


def shift(X, dx, dy):
    """Dời nét chữ dx điểm ảnh sang phải và dy điểm ảnh xuống dưới, điền 0 (giống convnet.shift của demo)."""
    out = np.zeros_like(X)
    H, W = X.shape[1:3]
    ys, yd = (slice(0, H - dy), slice(dy, H)) if dy >= 0 else (slice(-dy, H), slice(0, H + dy))
    xs, xd = (slice(0, W - dx), slice(dx, W)) if dx >= 0 else (slice(-dx, W), slice(0, W + dx))
    out[:, yd, xd] = X[:, ys, xs]
    return out


digit = Xte[:1, :, :, 0].astype(float)  # ảnh kiểm tra đầu tiên, (1, 28, 28)
for name, f in [("max pooling", pool2), ("average pooling", lambda x: pool2(x, "avg")), ("MaxBlurPool", maxblurpool2)]:
    print(f"{name:>15}: {digit.shape[1:]} → {f(digit).shape[1:]}")

Hình dưới áp bốn cách thu nhỏ cho một chữ số thật (hàng trên) và cho chính nó dời 1 điểm ảnh sang phải (hàng dưới).
Cột đầu là ảnh 28×28; bốn cột sau là bản 14×14: chỉ lấy mẫu thưa (một ô trong hai, như tích chập bước 2), max
pooling, average pooling và MaxBlurPool. Màu càng đậm thì giá trị càng lớn.

In [ ]:
DOWN = {
    "lấy mẫu thưa": subsample2,
    "max pooling": pool2,
    "average pooling": lambda x: pool2(x, "avg"),
    "MaxBlurPool": maxblurpool2,
}
j_demo = np.flatnonzero(yte == 7)[0]
pair = np.concatenate([Xte[j_demo : j_demo + 1, :, :, 0], shift(Xte[j_demo : j_demo + 1, :, :, 0], 1, 0)])
fig, axes = plt.subplots(2, 5, figsize=(10, 5.0))
for r, row_name in enumerate(["ảnh gốc", "dời 1 điểm ảnh →"]):
    axes[r, 0].imshow(pair[r], cmap="gray_r", vmin=0, vmax=1)
    axes[r, 0].set_title(f"{row_name}\n28×28", fontsize=10)
    for c, (name, f) in enumerate(DOWN.items(), start=1):
        axes[r, c].imshow(f(pair[r : r + 1])[0], cmap="gray_r", vmin=0, vmax=1)
        axes[r, c].set_title(f"{name}\n14×14", fontsize=10)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():  # khung xám đủ 4 cạnh: thấy rõ mép ảnh khi dời
        sp.set_visible(True)
        sp.set_color(GREY)
plt.tight_layout()
plt.show()

Ở hàng dưới, mỗi bản 14×14 đổi theo một kiểu riêng chứ không chỉ trượt đi: một nét dọc mảnh có thể rơi vào ô khác của
lưới 2×2, nên lúc đậm lúc nhạt. Ô dưới đo điều đó trên cả 10.000 ảnh kiểm tra, bằng độ thay đổi tương đối
$\lVert f(\text{dời}(x)) - f(x) \rVert / \lVert f(x) \rVert$:
- **dời 1 điểm ảnh:** so với đầu ra của ảnh gốc;
- **dời 2 điểm ảnh:** so với đầu ra của ảnh gốc đã dời 1 ô (14×14), tức là hỏi “đầu ra có dời theo đúng 1 ô không?”.

In [ ]:
X2 = Xte[:, :, :, 0].astype(float)  # 10.000 ảnh (N, 28, 28)


def rel_change(a, b):
    """Trung bình trên các ảnh của ‖a − b‖ / ‖b‖."""
    num = np.linalg.norm((a - b).reshape(len(a), -1), axis=1)
    return float(np.mean(num / np.linalg.norm(b.reshape(len(b), -1), axis=1)))


sens = {}
for name, f in DOWN.items():
    base = f(X2)
    sens[name] = (rel_change(f(shift(X2, 1, 0)), base), rel_change(f(shift(X2, 2, 0)), shift(base, 1, 0)))
    print(f"{name:>15}: dời 1 điểm ảnh → đổi {pct(sens[name][0], 1):>6}   dời 2 điểm ảnh → lệch {vn(sens[name][1], 4)}")
assert all(sens[n][1] < 1e-12 for n in ("lấy mẫu thưa", "max pooling", "average pooling"))
assert sens["MaxBlurPool"][0] < min(sens[n][0] for n in ("lấy mẫu thưa", "max pooling", "average pooling"))

Dời 2 điểm ảnh (một bội của bước), cả ba cách lấy bước 2 trực tiếp đều cho đúng đầu ra cũ dời 1 ô: độ lệch bằng 0.
Dời 1 điểm ảnh thì không cách nào giữ nguyên đầu ra: một bước 1 điểm ảnh ở ảnh 28×28 là nửa ô ở lưới 14×14, và không
lưới nào dời được nửa ô. Max pooling và average pooling đổi gần như nhau (46,5% và 47,2%), ít hơn lấy mẫu thưa.
MaxBlurPool đổi ít nhất, vì trên bản đồ đã làm mờ thì hai ô cạnh nhau có giá trị gần nhau, nhưng cũng không bằng 0.
Độ lệch rất nhỏ của nó khi dời 2 điểm ảnh đến từ viền: phần đệm ở mép không dời theo ảnh.

Max pooling chỉ chịu được những xê dịch nhỏ **một phần**: nếu nét lớn nhất vẫn nằm trong cùng ô 2×2 thì đầu ra không
đổi. Mục 6 đo xem điều đó có đủ để giữ nguyên dự đoán của cả mạng không.

## 4 · Công thức kích thước đầu ra

Một lớp tích chập hay pooling trượt cửa sổ $k \times k$ trên bản đồ cạnh $n_{\text{in}}$, đã đệm thêm $p$ ô mỗi bên,
mỗi lần nhảy $s$ ô. Số vị trí đặt được cửa sổ theo mỗi cạnh là

$$n_{\text{out}} = \left\lfloor \frac{n_{\text{in}} + 2p - k}{s} \right\rfloor + 1$$

Phần lẻ của phép chia bị làm tròn xuống: những hàng cuối không đủ cho thêm một bước thì bị bỏ, không bao giờ được bộ
lọc nhìn tới. Ô dưới kiểm tra công thức bằng cách *đếm thẳng* các vị trí bắt đầu của cửa sổ, với mọi tổ hợp
$n_{\text{in}} \le 40$, $k \in \{1, 2, 3, 5, 7, 11\}$, $s \le 4$, $p \le 3$, và bằng cỡ thật của mảng mà hàm
`patches()` (dùng cho tích chập ở mục 6) tạo ra.

In [ ]:
def n_out(n, k, s=1, p=0):
    return (n + 2 * p - k) // s + 1


def n_out_loop(n, k, s=1, p=0):
    """Đếm thẳng: cửa sổ bắt đầu ở ô start (tọa độ của ảnh đã đệm) và phải nằm gọn trong ảnh đã đệm."""
    return sum(1 for start in range(0, n + 2 * p, s) if start + k <= n + 2 * p)


def patches(x, k, s=1, p=0):
    """(N, H, W, C) → (N, Ho, Wo, k·k·C): mỗi ô đầu ra nhận vùng k×k×C của nó, theo thứ tự (hàng, cột, kênh)."""
    if p:
        x = np.pad(x, ((0, 0), (p, p), (p, p), (0, 0)))
    w = sliding_window_view(x, (k, k), axis=(1, 2))[:, ::s, ::s]  # (N, Ho, Wo, C, k, k)
    N, Ho, Wo, C = w.shape[:4]
    return w.transpose(0, 1, 2, 4, 5, 3).reshape(N, Ho, Wo, k * k * C)


cases = [(n, k, s, p) for n in range(1, 41) for k in (1, 2, 3, 5, 7, 11) for s in (1, 2, 3, 4) for p in (0, 1, 2, 3)]
cases = [c for c in cases if c[0] + 2 * c[3] >= c[1]]  # bộ lọc phải vừa trong ảnh đã đệm
ok = all(n_out(*c) == n_out_loop(*c) for c in cases)
ok_shape = all(patches(np.zeros((1, n, n, 1), F32), k, s, p).shape[1] == n_out(n, k, s, p) for n, k, s, p in cases)
print(f"{vn(len(cases), 0)} tổ hợp (n, k, s, p): công thức = đếm thẳng: {ok} · = cỡ mảng của patches(): {ok_shape}")
assert ok and ok_shape

Vài trường hợp quen thuộc, kèm số hàng bị bỏ vì làm tròn xuống, $(n_{\text{in}} + 2p - k) \bmod s$:

In [ ]:
for label, (n, k, s, p) in [
    ("tích chập 3×3, p = 1 (giữ nguyên cỡ)", (28, 3, 1, 1)),
    ("max pooling 2×2 bước 2", (28, 2, 2, 0)),
    ("max pooling 2×2 trên cạnh lẻ", (7, 2, 2, 0)),
    ("AlexNet lớp 1: 11×11 bước 4, ảnh 224", (224, 11, 4, 0)),
    ("AlexNet lớp 1: 11×11 bước 4, ảnh 227", (227, 11, 4, 0)),
]:
    print(f"{label:<38} ⌊({n} + 2·{p} − {k}) / {s}⌋ + 1 = {n_out(n, k, s, p):>3}   bỏ {(n + 2 * p - k) % s} hàng")
assert n_out(227, 11, 4) == 55 and (227 - 11) % 4 == 0 and (224 - 11) % 4 == 1

Bài báo AlexNet viết ảnh vào là 224×224, nhưng với bộ lọc 11×11 bước 4 thì $(224 - 11)/4$ không chia hết: ra 54 ô
và hàng cuối bị bỏ, trong khi bài báo vẽ bản đồ 55×55. Ảnh 227×227 mới cho đúng 55 (CS231n đã chỉ ra điểm này).
Pooling 2×2 trên cạnh lẻ 7 cũng bỏ hàng cuối: ra 3 ô, và hàng thứ 7 không thuộc ô nào.

## 5 · Vùng tiếp nhận lớn dần theo độ sâu

**Vùng tiếp nhận** (receptive field) của một ô là vùng ảnh vào có thể ảnh hưởng tới giá trị của ô đó. Một lớp
$k \times k$ bước 1 nới vùng ấy thêm $k - 1$ điểm ảnh. Sau một lớp bước $s$, hai ô cạnh nhau cách nhau $s$ lần xa hơn
trên ảnh vào, nên mỗi lớp sau nới vùng nhiều hơn. Gọi $j$ là khoảng cách (tính bằng điểm ảnh của ảnh vào) giữa hai ô
cạnh nhau. Bắt đầu từ $r = 1$, $j = 1$, qua mỗi lớp:

$$r \leftarrow r + (k - 1)\, j, \qquad j \leftarrow j \cdot s$$

(Araujo et al., 2019). Padding không đổi kích thước vùng, chỉ dời vị trí của nó. Ta mô tả một mạng bằng danh sách lớp;
hàm `rf()` chạy công thức trên. LeNet của demo có hai lớp tích chập 5×5 và hai lần pooling 2×2.

In [ ]:
LENET_SPATIAL = [("conv", 5, 1, 2), ("pool", 2, 2, 0), ("conv", 5, 1, 0), ("pool", 2, 2, 0)]  # (loại, k, s, p)


def rf(layers):
    """Vùng tiếp nhận r, khoảng cách j giữa hai ô kề nhau, và độ đệm hiệu dụng P (Araujo et al., 2019)."""
    r, j, P = 1, 1, 0
    for _, k, s, p in layers:
        r, P, j = r + (k - 1) * j, P + p * j, j * s
    return r, j, P


for depth in range(1, len(LENET_SPATIAL) + 1):
    name, k, s, p = LENET_SPATIAL[depth - 1]
    r, j, _ = rf(LENET_SPATIAL[:depth])
    print(f"sau lớp {depth} ({'tích chập' if name == 'conv' else 'pooling'} {k}×{k}, bước {s}): vùng {r}×{r}, j = {j}")

Kiểm tra trực tiếp, không dùng công thức: đưa vào 784 ảnh, mỗi ảnh chỉ sáng **một** điểm ảnh, qua đúng 4 lớp trên
với mọi trọng số bằng 1 (tích chập = tổng của cửa sổ, pooling = max). Giá trị đầu vào đều không âm, nên một ô đầu ra
khác 0 khi và chỉ khi điểm ảnh sáng nằm trong vùng tiếp nhận của nó. Theo Araujo et al., ô $u$ (đếm từ 0) nhìn từ
điểm ảnh $u \cdot j - P$ đến $u \cdot j - P + r - 1$, với $P$ là độ đệm hiệu dụng mà `rf()` trả về.

In [ ]:
def influence(layers, n=28):
    """Với mỗi điểm ảnh vào: những ô nào của lớp cuối nhận được ảnh hưởng của nó? → (n·n, n_out, n_out) kiểu bool."""
    x = np.eye(n * n, dtype=F32).reshape(n * n, n, n, 1)  # ảnh thứ i chỉ sáng điểm ảnh i
    for kind, k, s, p in layers:
        win = patches(x, k, s, p)  # (N, Ho, Wo, k·k)
        x = (win.sum(-1) if kind == "conv" else win.max(-1))[..., None]
    return x[..., 0] > 0


r, j, P = rf(LENET_SPATIAL)
hit = influence(LENET_SPATIAL)
print(f"lớp cuối: {hit.shape[1]}×{hit.shape[2]} ô; công thức: r = {r}, j = {j}, P = {P}")
for u in range(hit.shape[1]):
    mask = hit[:, u, u].reshape(28, 28)
    rows = np.flatnonzero(mask.any(1))
    lo, hi = u * j - P, u * j - P + r - 1
    print(
        f"ô ({u}, {u}): ảnh hưởng từ hàng {rows.min()} đến {rows.max()}; công thức: {vn(lo, 0)} đến {hi} (cắt vào ảnh 0…27)"
    )
    box = np.zeros((28, 28), bool)
    box[max(lo, 0) : hi + 1, max(lo, 0) : hi + 1] = True
    assert np.array_equal(mask, box), "vùng tiếp nhận đếm trực tiếp khác công thức"

Đếm trực tiếp khớp công thức ở mọi ô. Ô giữa, (2, 2), nhìn đúng một vùng 16×16; ô ở góc, (0, 0), có vùng bắt đầu từ
hàng −2, nên 2 hàng và 2 cột của nó rơi vào phần đệm 0. Hình dưới vẽ hai vùng đó trên một chữ số: ô tô xanh là những
điểm ảnh mà phép đếm trực tiếp tìm thấy, khung gạch là vùng $r \times r$ theo công thức (khung của ô góc thò ra ngoài
ảnh, vào phần đệm).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3.6))
for ax, u in zip(axes, (0, 2)):
    mask = hit[:, u, u].reshape(28, 28)
    ax.imshow(Xte[j_demo, :, :, 0], cmap="gray_r", vmin=0, vmax=1)
    ax.imshow(np.where(mask, 1.0, np.nan), cmap=LinearSegmentedColormap.from_list("x", [BLUE, BLUE]), alpha=0.35)
    lo = u * j - P
    ax.add_patch(Rectangle((lo - 0.5, lo - 0.5), r, r, fill=False, ls="--", lw=2, ec="black", clip_on=False))
    ax.set(xlim=(-3.5, 27.5), ylim=(27.5, -3.5), xticks=[], yticks=[])
    ax.set_title(f"ô ({u}, {u}) sau lớp 4: vùng {r}×{r}", fontsize=10)
plt.tight_layout()
plt.show()

### Hai lớp 3×3 hay một lớp 5×5?

Theo công thức, hai lớp 3×3 bước 1 có vùng $1 + 2 + 2 = 5$, bằng một lớp 5×5; ba lớp 3×3 bằng một lớp 7×7. Nhưng số
trọng số thì khác. Với $C$ kênh vào và $C$ kênh ra, một lớp $k \times k$ có $k^2 C^2$ trọng số (bỏ qua độ lệch), nên:

In [ ]:
for n_layers, k_big in [(2, 5), (3, 7)]:
    r_stack, _, _ = rf([("conv", 3, 1, 1)] * n_layers)
    w_stack, w_big = n_layers * 3 * 3, k_big * k_big
    print(
        f"{n_layers} lớp 3×3: vùng {r_stack}×{r_stack}, {w_stack}C² trọng số · 1 lớp {k_big}×{k_big}: {w_big}C²"
        f"  (nhiều hơn {pct(w_big / w_stack - 1, 0)}) · với C = 512: {vn(w_stack * 512**2, 0)} so với {vn(w_big * 512**2, 0)}"
    )
    assert r_stack == k_big
assert (2 * 9, 25, 3 * 9, 49) == (18, 25, 27, 49) and round(100 * (49 / 27 - 1)) == 81

Đó là lý do VGG (Simonyan & Zisserman, 2014) chỉ dùng bộ lọc 3×3: cùng vùng nhìn, ít trọng số hơn, và thêm một ReLU
giữa mỗi hai lớp. Bài báo viết đúng con số này: một lớp 7×7 cần $49C^2$ tham số, “nhiều hơn 81%” so với $27C^2$ của ba
lớp 3×3.

### Vùng tiếp nhận của VGG‑16 và AlexNet
VGG‑16 chỉ dùng bộ lọc 3×3, cùng 5 lần max pooling 2×2. Bộ lọc nhỏ không có nghĩa là nhìn hẹp: mỗi lần pooling nhân
đôi $j$, nên các lớp 3×3 ở sâu nới vùng rất nhanh. Ô dưới tính vùng tiếp nhận của một ô trong bản đồ 7×7×512 sau lần
pooling cuối, và của AlexNet với các bộ lọc lớn 11×11 và 5×5.

In [ ]:
VGG16_SPATIAL = []
for n_conv in (2, 2, 3, 3, 3):  # 5 khối, mỗi khối vài lớp 3×3 rồi max pooling 2×2
    VGG16_SPATIAL += [("conv", 3, 1, 1)] * n_conv + [("pool", 2, 2, 0)]
ALEXNET_SPATIAL = [("conv", 11, 4, 0), ("pool", 3, 2, 0), ("conv", 5, 1, 2), ("pool", 3, 2, 0)]
ALEXNET_SPATIAL += [("conv", 3, 1, 1)] * 3 + [("pool", 3, 2, 0)]
for name, net in [("VGG‑16", VGG16_SPATIAL), ("AlexNet", ALEXNET_SPATIAL)]:
    r_, j_, P_ = rf(net)
    print(f"{name:>8}: {len(net)} lớp không gian → vùng tiếp nhận {r_}×{r_}, bước giữa hai ô j = {j_}, độ đệm P = {P_}")
# Araujo et al. (2019), bảng các mạng phổ biến: vgg_16 (r, S, P) = (212, 32, 90), alexnet_v2 = (195, 32, 64)
# (alexnet_v2 của TF-Slim có cùng các lớp không gian: k, bước và padding như trên)
assert rf(VGG16_SPATIAL) == (212, 32, 90) and rf(ALEXNET_SPATIAL) == (195, 32, 64)

Sau 13 lớp 3×3 và 5 lần pooling, một ô của VGG‑16 nhìn vùng 212×212, gần trọn ảnh 224×224. AlexNet đạt 195×195 với
8 lớp không gian. Cả hai khớp bảng của Araujo et al. (2019), mà ô trên dùng làm `assert`. Bước giữa hai ô kề nhau là
$2^5 = 32$ điểm ảnh: bản đồ cuối là một lưới thô, mỗi ô nhìn một vùng rộng.

Vùng tiếp nhận chỉ cho biết điểm ảnh nào *có thể* ảnh hưởng tới ô. Điểm ảnh ở giữa vùng có nhiều đường đi tới ô hơn
điểm ở mép, nên ảnh hưởng nhiều hơn hẳn; Luo et al. (2016) gọi phần thực sự quan trọng là *vùng tiếp nhận hiệu dụng*
và đo được nó có dạng gần như hình chuông, nhỏ hơn nhiều so với $r \times r$.

## 6 · LeNet trên MNIST: huấn luyện và dời ảnh

Ghép các mảnh lại. LeNet của demo (cho ảnh 28×28) gồm:

tích chập 5×5 × 6, đệm 2 → ReLU → pooling 2×2 → tích chập 5×5 × 16 → ReLU → pooling 2×2 → dense 400 → 120 → 84 → 10.

Tích chập là phép nhân ma trận: `patches()` gom vùng $k \times k \times C$ của mỗi ô thành một hàng, rồi nhân với
ma trận trọng số $W$ cỡ $(k^2 C) \times C_{\text{out}}$, như $z = xW + b$ của một lớp dense. Ô dưới kiểm tra điều đó
với một vòng lặp tường minh trên một ảnh, rồi viết lan truyền ngược của từng lớp. Pooling max chuyển gradient về đúng
ô lớn nhất của mỗi ô 2×2 (chia đều nếu có nhiều ô bằng nhau, như `convnet.Pool` của demo).

In [ ]:
def unpatches(d, shape, k, p=0):
    """Ngược của patches() (bước 1): cộng gradient của mỗi hàng về đúng vùng k×k×C của nó."""
    N, H, W, C = shape
    Ho, Wo = d.shape[1:3]
    d = d.reshape(N, Ho, Wo, k, k, C)
    dx = np.zeros((N, H + 2 * p, W + 2 * p, C), F32)
    for i in range(k):
        for j in range(k):
            dx[:, i : i + Ho, j : j + Wo] += d[:, :, :, i, j]
    return dx[:, p : p + H, p : p + W] if p else dx


def pool_fwd(x, kind):
    """Pooling 2×2 bước 2 trên (N, H, W, C), cạnh chẵn; trả về đầu ra và những gì cần cho lan truyền ngược."""
    y = pool2(x, kind)
    if kind == "avg":
        return y, None
    eq = [x[:, i::2, j::2] == y for i, j in QUAD]  # ô nào là max (có thể nhiều ô bằng nhau)
    return y, (eq, eq[0].astype(F32) + eq[1] + eq[2] + eq[3])


def pool_bwd(dy, cache, kind):
    N, a, b, C = dy.shape
    dx = np.empty((N, 2 * a, 2 * b, C), F32)
    for t, (i, j) in enumerate(QUAD):
        dx[:, i::2, j::2] = dy / 4 if kind == "avg" else cache[0][t] * (dy / cache[1])
    return dx


# Kiểm tra: tích chập bằng vòng lặp = patches() @ W, trên ảnh kiểm tra đầu tiên
rng = np.random.default_rng(0)
W_try = rng.normal(0, 0.2, (5 * 5 * 1, 6)).astype(F32)
img = Xte[:1]
loop = np.zeros((28, 28, 6), F32)
padded = np.pad(img[0], ((2, 2), (2, 2), (0, 0)))
for u in range(28):
    for v in range(28):
        loop[u, v] = padded[u : u + 5, v : v + 5, :].reshape(-1) @ W_try  # 25 MAC cho mỗi bộ lọc
print("vòng lặp = patches() @ W:", np.allclose(loop, (patches(img, 5, 1, 2) @ W_try)[0], atol=1e-5))
assert np.allclose(loop, (patches(img, 5, 1, 2) @ W_try)[0], atol=1e-5)

Mạng hoàn chỉnh. Khởi tạo He $\mathcal{N}(0, 2/\text{fan-in})$ cho mọi trọng số, độ lệch bằng 0, rút từ
`default_rng(seed)` theo đúng thứ tự lớp như script `lenet.py` của demo; mỗi lượt (epoch) xáo thứ tự ảnh bằng cùng bộ
sinh đó. Huấn luyện bằng SGD $\eta = 0{,}1$, lô 64 ảnh, mất mát cross-entropy (Bài 04).

In [ ]:
SHAPES = {"W1": (25, 6), "W2": (150, 16), "W3": (400, 120), "W4": (120, 84), "W5": (84, 10)}  # (fan-in, fan-out)


def init_lenet(seed):
    rng = np.random.default_rng(seed)
    P = {}
    for name, (fan_in, fan_out) in SHAPES.items():
        P[name] = rng.normal(0, np.sqrt(2 / fan_in), (fan_in, fan_out)).astype(F32)
        P["b" + name[1:]] = np.zeros(fan_out, F32)
    return P, rng


def forward(P, x, kind="max", c=None):
    """x: (N, 28, 28, 1) → điểm số (N, 10). c (nếu có) giữ lại các giá trị trung gian cho lan truyền ngược."""
    x1 = patches(x, 5, 1, 2)  # (N, 28, 28, 25)
    a1 = x1 @ P["W1"] + P["b1"]  # (N, 28, 28, 6)
    p1, m1 = pool_fwd(np.maximum(a1, 0), kind)  # (N, 14, 14, 6)
    x2 = patches(p1, 5)  # (N, 10, 10, 150)
    a2 = x2 @ P["W2"] + P["b2"]  # (N, 10, 10, 16)
    p2, m2 = pool_fwd(np.maximum(a2, 0), kind)  # (N, 5, 5, 16)
    f = p2.reshape(len(x), 400)
    h3 = np.maximum(f @ P["W3"] + P["b3"], 0)
    h4 = np.maximum(h3 @ P["W4"] + P["b4"], 0)
    if c is not None:
        c.update(x1=x1, a1=a1, m1=m1, x2=x2, a2=a2, m2=m2, f=f, h3=h3, h4=h4)
    return h4 @ P["W5"] + P["b5"]


def sgd_step(P, x, y, kind="max", eta=0.1):
    c = {}
    z = forward(P, x, kind, c)
    p = np.exp(z - z.max(1, keepdims=True))
    p /= p.sum(1, keepdims=True)
    p[np.arange(len(y)), y] -= 1
    d = (p / len(y)).astype(F32)  # ∂L/∂z của cross-entropy trung bình
    g = {"W5": c["h4"].T @ d, "b5": d.sum(0)}
    d = (d @ P["W5"].T) * (c["h4"] > 0)
    g["W4"], g["b4"] = c["h3"].T @ d, d.sum(0)
    d = (d @ P["W4"].T) * (c["h3"] > 0)
    g["W3"], g["b3"] = c["f"].T @ d, d.sum(0)
    d = pool_bwd((d @ P["W3"].T).reshape(-1, 5, 5, 16), c["m2"], kind) * (c["a2"] > 0)
    g["W2"], g["b2"] = c["x2"].reshape(-1, 150).T @ d.reshape(-1, 16), d.sum((0, 1, 2))
    d = pool_bwd(unpatches(d @ P["W2"].T, (len(y), 14, 14, 6), 5), c["m1"], kind) * (c["a1"] > 0)
    g["W1"], g["b1"] = c["x1"].reshape(-1, 25).T @ d.reshape(-1, 6), d.sum((0, 1, 2))
    for k in P:
        P[k] -= eta * g[k]


def predict(P, X, kind="max"):
    return np.concatenate([forward(P, X[i : i + 1000], kind).argmax(1) for i in range(0, len(X), 1000)])


def train_lenet(kind="max", seed=0, steps=500, batch=64):
    P, rng = init_lenet(seed)
    order = rng.permutation(len(Xtr))  # lượt 1 của lenet.py: cùng trọng số đầu, cùng thứ tự ảnh
    for t in range(steps):
        idx = order[t * batch : (t + 1) * batch]
        sgd_step(P, Xtr[idx], ytr[idx], kind)
    return P


n_params = sum(v.size for v in init_lenet(0)[0].values())
print(f"LeNet cho MNIST: {vn(n_params, 0)} tham số")
assert n_params == 61706 == lenet_runs["runs"]["max"]["params"]

Demo huấn luyện mạng này 5 lượt trên cả 60.000 ảnh; như thế mất vài phút với numpy. Ta chạy ngắn hơn: 600 lô đầu
của lượt 1, với đúng trọng số khởi tạo và thứ tự ảnh của seed 0.

### 🤔 Dự đoán trước
Mạng có hai lần max pooling. Sau khi huấn luyện, ta dời **mọi** ảnh kiểm tra đi 1 điểm ảnh (sang phải, trái, xuống hoặc
lên; chỗ trống điền 0). Bao nhiêu phần trăm dự đoán thay đổi: 0%, vài phần trăm, hay hơn một nửa?

In [ ]:
STEPS = 600
t0 = time.perf_counter()
P_max = train_lenet("max", seed=0, steps=STEPS)
pred0 = predict(P_max, Xte)
print(f"{STEPS} bước SGD, {vn(STEPS * 64, 0)} ảnh ({vn(time.perf_counter() - t0, 1)} giây)")
print(f"đúng {pct((pred0 == yte).mean())} ảnh kiểm tra")

DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]  # (dx, dy): phải, trái, xuống, lên
moved = {d: predict(P_max, shift(Xte, *d)) for d in DIRS}
changed = {d: moved[d] != pred0 for d in DIRS}
change1 = np.mean([changed[d].mean() for d in DIRS])  # trung bình 4 hướng, như demo
acc1 = np.mean([(moved[d] == yte).mean() for d in DIRS])
print(f"dời 1 điểm ảnh: {pct(change1)} dự đoán thay đổi (trung bình 4 hướng); còn đúng {pct(acc1)}")
print(
    " · ".join(f"{name} {pct(changed[d].mean())}" for name, d in zip(["phải", "trái", "xuống", "lên"], DIRS)),
    f"· đổi ở ít nhất một hướng: {vn(np.any(list(changed.values()), axis=0).sum(), 0)} ảnh",
)
assert (pred0 == yte).mean() > 0.95 and change1 > 0.005

Vài phần trăm, không phải 0. Max pooling không làm mạng **bất biến** với phép dời: như ở mục 3, dời 1 điểm ảnh là dời
nửa ô của lưới sau lần pooling đầu và một phần tư ô sau lần thứ hai, nên các đặc trưng không chỉ trượt đi mà đổi giá
trị. Với những ảnh nằm sát ranh giới giữa hai lớp, chừng đó là đủ để đổi câu trả lời. Azulay & Weiss (2019) đo điều
tương tự trên các mạng ImageNet: dời ảnh xuống 1 điểm ảnh đổi tới 30% dự đoán, dù các mạng ấy được huấn luyện với
tăng cường dữ liệu.

Hàng trên của hình dưới là 8 ảnh kiểm tra đầu tiên có dự đoán đổi khi dời sang phải 1 điểm ảnh, hàng dưới là chúng
sau khi dời; tiêu đề ghi nhãn đúng và dự đoán.

In [ ]:
flip = np.flatnonzero(changed[(1, 0)])[:8]
right = shift(Xte[flip], 1, 0)
fig, axes = plt.subplots(2, 8, figsize=(10, 3.1))
for c, i in enumerate(flip):
    for r, (img_, p) in enumerate([(Xte[i], pred0[i]), (right[c], moved[(1, 0)][i])]):
        axes[r, c].imshow(img_[:, :, 0], cmap="gray_r", vmin=0, vmax=1)
        axes[r, c].set_title(f"{'gốc' if r == 0 else 'dời →'}: đoán {p}", fontsize=9)
        axes[r, c].set_xticks([])
        axes[r, c].set_yticks([])
    axes[0, c].set_xlabel(f"nhãn {yte[i]}", fontsize=9)
plt.tight_layout()
plt.show()

## 7 · Kết quả đầy đủ: sáu cách thu nhỏ, 3 seed

Lần chạy trên ngắn (600 bước) và chỉ một seed. Script `lenet.py` của demo huấn luyện cùng mạng đủ 5 lượt trên
60.000 ảnh với seed 0, 1, 2, cho sáu biến thể:
- **max pooling**, **average pooling**, **MaxBlurPool**: ba cách pooling ở mục 3;
- **stride 2**: bỏ pooling, cho hai lớp tích chập bước 2, nên bản đồ vẫn nhỏ đi (chỉ lấy mẫu thưa, không gộp);
- **không pooling**: bản đồ giữ cỡ 24×24, nên lớp dense đầu nhận 9.216 giá trị;
- **Bài 11**: CNN một lớp của bài trước (tích chập 3×3 × 8, một lần max pooling).

Mỗi mạng được thử trên ảnh kiểm tra dời $k = 1 \dots 4$ điểm ảnh, lấy trung bình bốn hướng như ở mục 6. Chạy lại mất
vài phút, nên ta đọc kết quả từ tệp của demo. Mỗi số dưới đây là khoảng min–max của 3 seed.

In [ ]:
VARIANTS = {"max": "max pooling", "avg": "average pooling", "blur": "MaxBlurPool"}
VARIANTS |= {"stride": "stride 2", "none": "không pooling", "bai11": "Bài 11"}
K = lenet_runs["shift"]
runs = lenet_runs["runs"]
print(
    f"{len(runs)} mạng × {len(lenet_runs['seeds'])} seed, {lenet_runs['epochs']} lượt, η = {vn(lenet_runs['eta'], 1)}"
)


def spread(v, f=pct, d=1):
    return f"{f(min(v), d)}–{f(max(v), d)}"


def stats(rid):
    r = runs[rid]
    return {
        "test": [a[0] for a in r["acc"]],
        "chg1": [1 - s[1] for s in r["same"]],
        "acc4": [a[K] for a in r["acc"]],
    }


S = {rid: stats(rid) for rid in VARIANTS}
for rid, name in VARIANTS.items():
    s = S[rid]
    print(
        f"{name:>15}: {vn(runs[rid]['params'], 0):>9} tham số · đúng {spread(s['test'], d=2)} · "
        f"dời 1 điểm ảnh đổi {spread(s['chg1'])} · dời {K} điểm ảnh còn đúng {spread(s['acc4'])}"
    )

Hình dưới vẽ hai cột của bảng. Mỗi hàng là một mạng, có nhãn tên ở bên trái; ba chấm là ba seed, vạch ngang nối min
với max, vạch đứng là trung vị. Hàng max pooling tô xanh. Bên trái: tỉ lệ dự đoán đổi khi dời 1 điểm ảnh (càng thấp
càng ổn định). Bên phải: tỉ lệ còn đúng khi dời 4 điểm ảnh (càng cao càng tốt).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
names = list(VARIANTS)
for ax, key, title in [
    (axes[0], "chg1", "Dự đoán đổi khi dời 1 điểm ảnh (%)"),
    (axes[1], "acc4", f"Còn đúng khi dời {K} điểm ảnh (%)"),
]:
    for row, rid in enumerate(names):
        v = 100 * np.array(S[rid][key])
        col = BLUE if rid == "max" else "black"
        ax.plot([v.min(), v.max()], [row, row], color=col, lw=2, alpha=0.6)
        ax.plot(v, [row] * 3, "o", color=col, ms=6, mfc="white" if rid != "max" else BLUE)
        ax.plot([np.median(v)] * 2, [row - 0.25, row + 0.25], color=col, lw=2)
    ax.set_title(title, fontsize=10)
    ax.grid(axis="x", alpha=0.3)
axes[0].set_yticks(range(len(names)), [VARIANTS[r] for r in names])
axes[0].invert_yaxis()
vn_axes(fig, "x")
plt.tight_layout()
plt.show()

Ô dưới so từng mạng với LeNet **không pooling**, theo quy tắc của demo: hai mạng chỉ khác nhau rõ khi khoảng 3 seed
của chúng không chồng lên nhau.

In [ ]:
def overlap(a, b):
    return max(min(a), min(b)) <= min(max(a), max(b))


def verdict(a, b, high_good):
    if overlap(a, b):
        return "ngang nhau (trong dao động giữa các seed)"
    return "tốt hơn rõ" if (np.median(a) > np.median(b)) == high_good else "kém hơn rõ"


for rid in ["max", "avg", "blur", "stride"]:
    print(
        f"{VARIANTS[rid]:>15} so với không pooling: dời 1 điểm ảnh {verdict(S[rid]['chg1'], S['none']['chg1'], False)}; "
        f"dời {K} điểm ảnh {verdict(S[rid]['acc4'], S['none']['acc4'], True)}"
    )
print(f"MaxBlurPool so với max pooling, dời {K} điểm ảnh: {verdict(S['blur']['acc4'], S['max']['acc4'], True)}")
lo, hi = min(S["max"]["chg1"]), max(S["max"]["chg1"])
print(
    f"lần chạy ngắn của ta (mục 6): {pct(change1)} đổi; trong khoảng 3 seed của max pooling ({spread(S['max']['chg1'])}):",
    lo - 0.005 <= change1 <= hi + 0.005,
)
best4 = max(VARIANTS, key=lambda r: np.median(S[r]["acc4"]))
ratio = runs["none"]["params"] / runs["max"]["params"]
print(f"dời {K} điểm ảnh, trung vị cao nhất: {VARIANTS[best4]} {pct(np.median(S[best4]['acc4']), 1)}")
print(f"không pooling / max pooling: {vn(ratio, 1)} lần nhiều tham số")

Những điều đáng nhớ:
- **Dời 1 điểm ảnh:** mọi mạng đều đổi một phần dự đoán, kể cả mạng không có pooling nào (1,8–2,2%). Ba cách pooling
  ngang với không pooling, trong dao động giữa các seed; chỉ stride 2 kém hơn rõ.
- **Dời 4 điểm ảnh:** cả ba cách pooling giữ được nhiều hơn rõ so với không pooling, còn stride 2 kém hơn rõ. Gộp các ô
  lân cận (lấy max, lấy trung bình, làm mờ) giúp chịu được xê dịch một phần; chỉ lấy mẫu thưa thì không. MaxBlurPool có
  trung vị cao nhất, nhưng khoảng seed của nó chồng với max pooling.
- **Kích thước:** không pooling thì bản đồ không nhỏ lại, lớp dense đầu nhận 9.216 giá trị và mạng có 1.119.626 tham
  số, gấp 18,1 lần, mà khi dời xa còn kém hơn.
- **Không mạng nào bất biến:** dời 4 điểm ảnh, mạng tốt nhất (MaxBlurPool) chỉ còn đúng 62,7% (trung vị), trong khi
  trên ảnh không dời nó đúng 98,01–99,05%.

Muốn mạng chịu được phép dời tốt hơn thì phải dạy nó: tăng cường dữ liệu bằng ảnh dời (Bài 11 đã làm với MLP), hoặc
lấy mẫu chống aliasing như MaxBlurPool. Azulay & Weiss (2019) cho thấy cả hai đều chỉ giải quyết được một phần.

## 8 · Đếm tham số và phép tính, từng lớp một

Chỉ cần công thức kích thước ở mục 4 là đếm được mọi thứ của một CNN mà không phải chạy nó. Với một lớp tích chập
$k \times k$ từ $C_{\text{in}}$ sang $C_{\text{out}}$ kênh, ra bản đồ $n_{\text{out}} \times n_{\text{out}}$:
- **tham số:** $k^2 C_{\text{in}} C_{\text{out}} + C_{\text{out}}$ (trọng số dùng chung cho mọi vị trí, cộng một độ
  lệch mỗi bộ lọc);
- **phép nhân–cộng (MAC):** $n_{\text{out}}^2 \cdot k^2 C_{\text{in}} C_{\text{out}}$, vì mỗi trọng số được dùng lại ở
  mọi vị trí.

Lớp dense từ $n_{\text{in}}$ sang $n_{\text{out}}$ có $n_{\text{in}} n_{\text{out}} + n_{\text{out}}$ tham số và
$n_{\text{in}} n_{\text{out}}$ MAC: mỗi trọng số chỉ dùng một lần. Như demo, ta chỉ đếm phép nhân với trọng số (pooling,
ReLU, độ lệch không tính). Hàm `count()` dưới đây mô tả mạng bằng một danh sách lớp, giống ô “Lắp một mạng CNN” của
demo, và in bảng từng lớp.

In [ ]:
def conv(k, s, p, c, groups=1, pairs=None):
    return {"t": "conv", "k": k, "s": s, "p": p, "c": c, "g": groups, "pairs": pairs}


def pool(k=2, s=2):
    return {"t": "pool", "k": k, "s": s, "p": 0}


def fc(c, fixed=False):
    return {"t": "fc", "c": c, "fixed": fixed}


# “subsampling” của LeNet‑5: cộng ô 2×2, nhân một hệ số học được, cộng độ lệch (2 tham số mỗi bản đồ)
SUB = {"t": "sub"}
BLUR = {"t": "blur"}  # MaxBlurPool: ⌈n/2⌉, không có tham số
BN = {"t": "bn"}  # BatchNorm (Bài 10): γ và β cho mỗi kênh, không đổi cỡ
GAP = {"t": "gap"}  # pooling trung bình toàn cục: n × n × c → 1 × 1 × c


def count(n, c, layers, show=False):
    """Đi qua từng lớp: cỡ đầu ra, tham số, MAC. Trả về danh sách hàng (loại, n, c, tham số, MAC)."""
    rows, flat = [], False
    for L in layers:
        if L["t"] == "fc":
            n_in = c if flat else n * n * c
            params, macs = (0 if L["fixed"] else n_in * L["c"] + L["c"]), n_in * L["c"]
            n, c, flat = 1, L["c"], True
        elif L["t"] == "bn":
            params, macs = 2 * c, 0
        elif L["t"] == "gap":
            n, params, macs = 1, 0, 0
        elif L["t"] in ("sub", "blur"):
            n = n // 2 if L["t"] == "sub" else -(-n // 2)
            params, macs = (2 * c, n * n * c) if L["t"] == "sub" else (0, 0)
        else:
            n = n_out(n, L["k"], L["s"], L["p"])
            params = macs = 0
            if L["t"] == "conv":
                per = L["k"] ** 2 * (L["pairs"] if L["pairs"] else c // L["g"] * L["c"])  # trọng số mỗi vị trí
                params, macs, c = per + L["c"], n * n * per, L["c"]
        rows.append((L["t"], n, c, params, macs))
        if show:
            shape = f"{c}" if flat else f"{n}×{n}×{c}"
            print(f"  {L['t']:<5} → {shape:<12} {vn(params, 0):>12} tham số {vn(macs, 0):>16} MAC")
    return rows


def totals(rows):
    P, M = sum(r[3] for r in rows), sum(r[4] for r in rows)
    Pd, Md = sum(r[3] for r in rows if r[0] == "fc"), sum(r[4] for r in rows if r[0] == "fc")
    return P, M, Pd / P, Md / M

Trước hết là sáu mạng MNIST của mục 7. Số tham số mà `count()` tính phải bằng đúng số tham số của các mạng mà demo đã
huấn luyện (và bằng `n_params` của mạng ta tự viết ở mục 6):

In [ ]:
HEAD = [fc(120), fc(84), fc(10)]
MNIST_NETS = {
    "max": [conv(5, 1, 2, 6), pool(), conv(5, 1, 0, 16), pool(), *HEAD],
    "avg": [conv(5, 1, 2, 6), pool(), conv(5, 1, 0, 16), pool(), *HEAD],
    "blur": [conv(5, 1, 2, 6), BLUR, conv(5, 1, 0, 16), BLUR, *HEAD],
    "stride": [conv(5, 2, 2, 6), conv(5, 2, 0, 16), *HEAD],
    "none": [conv(5, 1, 2, 6), conv(5, 1, 0, 16), *HEAD],
    "bai11": [conv(3, 1, 0, 8), pool(), fc(10)],
}
print("LeNet với max pooling, từng lớp:")
count(28, 1, MNIST_NETS["max"], show=True)
for rid, layers in MNIST_NETS.items():
    P_, M_, pd, md = totals(count(28, 1, layers))
    print(
        f"{VARIANTS[rid]:>15}: {vn(P_, 0):>9} tham số, {vn(M_, 0):>10} MAC · dense: {pct(pd, 1)} tham số, {pct(md, 1)} MAC"
    )
    assert P_ == runs[rid]["params"], "khác với các mạng mà demo đã huấn luyện"
assert totals(count(28, 1, MNIST_NETS["max"]))[0] == n_params == 61706
assert totals(count(28, 1, MNIST_NETS["none"]))[0] == 1119626
_, n_none, c_none, _, _ = count(28, 1, MNIST_NETS["none"])[1]
print(f"không pooling: lớp dense đầu nhận {n_none}×{n_none}×{c_none} = {vn(n_none**2 * c_none, 0)} giá trị")

Stride 2 có đúng số tham số của bản max pooling (cùng các bộ lọc, chỉ khác chỗ thu nhỏ) nhưng ít phép tính hơn nhiều,
vì tích chập chỉ được tính ở một vị trí trong bốn. Bỏ pooling thì ngược lại: các lớp tích chập vẫn giữ nguyên số
tham số, nhưng lớp dense đầu phải nối với 24×24×16 giá trị, và mạng phình lên gấp 18,1 lần.

### LeNet‑5 (LeCun et al., 1998)

Bản gốc khác bản của demo ở bốn chỗ: ảnh vào 32×32 (chữ số 28×28 được đệm thêm), không có padding, thu nhỏ bằng
**subsampling** (cộng 4 ô, nhân một hệ số học được, cộng độ lệch: 2 tham số mỗi bản đồ) và lớp C3 chỉ nối 60 cặp bản đồ
(mỗi bản đồ của C3 nhìn 3, 4 hoặc 6 trong 6 bản đồ của S2; Bảng 1 của bài báo). Lớp ra là 10 đơn vị RBF có vectơ tham
số đặt bằng tay và giữ cố định (“at least initially”). Bài báo ghi số tham số học được của từng lớp, nên ta so được từng lớp một.

In [ ]:
C3_PAIRS = 6 * 3 + 6 * 4 + 3 * 4 + 1 * 6  # 6 bản đồ nhìn 3 bản đồ của S2, 6 + 3 bản đồ nhìn 4, 1 bản đồ nhìn cả 6
LENET5 = [conv(5, 1, 0, 6), SUB, conv(5, 1, 0, 16, pairs=C3_PAIRS), SUB, conv(5, 1, 0, 120), fc(84), fc(10, fixed=True)]
names5 = ["C1", "S2", "C3", "S4", "C5", "F6", "ra (RBF)"]
print(f"C3 nối {C3_PAIRS} cặp bản đồ")
rows5 = count(32, 1, LENET5, show=True)
paper = [156, 12, 1516, 32, 48120, 10164, 0]  # LeCun et al. (1998), mục II‑B
for name, r, want in zip(names5, rows5, paper):
    print(f"{name:>9}: {vn(r[3], 0):>6} tham số (bài báo: {vn(want, 0)})")
P5 = totals(rows5)[0]
print(f"tổng: {vn(P5, 0)} tham số học được; C3 nối đầy đủ thì có {vn(5 * 5 * 6 * 16 + 16, 0)}")
assert [r[3] for r in rows5] == paper and P5 == 60000 and 5 * 5 * 6 * 16 + 16 == 2416

Tổng đúng 60.000, con số bài báo ghi cho LeNet‑5 (“345 308 connections, but only 60 000 trainable free parameters”).
Nếu C3 nối đủ 6 × 16 cặp, riêng lớp đó đã có 2.416 thay vì 1.516 tham số.

### AlexNet (Krizhevsky, Sutskever & Hinton, 2012)

Năm lớp tích chập (11×11 bước 4, 5×5, ba lớp 3×3), max pooling 3×3 bước 2 (các ô chồng lên nhau), rồi ba lớp dense
4096 → 4096 → 1000. Mạng được chia cho **2 GPU** GTX 580 3 GB: ở các lớp tích chập 2, 4 và 5, mỗi nửa số bộ lọc chỉ nối
với nửa số kênh nằm trên cùng GPU (`groups=2`), nên các lớp này có một nửa số trọng số so với nối đầy đủ. Ảnh vào lấy
227×227 như ở mục 4.

### 🤔 Dự đoán trước
AlexNet có khoảng 60 triệu tham số. Ba lớp dense ở cuối chiếm bao nhiêu phần trăm số **tham số**? Và bao nhiêu phần
trăm số **phép nhân–cộng**: cũng chừng đó, hay khác hẳn?

In [ ]:
ALEXNET = [conv(11, 4, 0, 96), pool(3, 2), conv(5, 1, 2, 256, groups=2), pool(3, 2), conv(3, 1, 1, 384)]
ALEXNET += [conv(3, 1, 1, 384, groups=2), conv(3, 1, 1, 256, groups=2), pool(3, 2), fc(4096), fc(4096), fc(1000)]
rows_a = count(227, 3, ALEXNET, show=True)
Pa, Ma, pa, ma = totals(rows_a)
print(f"AlexNet: {vn(Pa, 0)} tham số, {vn(Ma, 0)} MAC cho một ảnh")
print(f"3 lớp dense: {pct(pa, 1)} tham số nhưng chỉ {pct(ma, 1)} phép nhân–cộng")
ungrouped = totals(count(227, 3, [{**L, "g": 1} if L["t"] == "conv" else L for L in ALEXNET]))[0]
print(f"nếu không chia 2 GPU: {vn(ungrouped, 0)} tham số")
neurons = [r[1] * r[1] * r[2] for r in rows_a if r[0] == "conv"]
paper_neurons = [253440, 186624, 64896, 64896, 43264]  # chú thích Hình 2 của bài báo
print("số nơ-ron của 5 lớp tích chập:", " – ".join(vn(v, 0) for v in neurons))
print("bài báo (Hình 2):             ", " – ".join(vn(v, 0) for v in paper_neurons))
print(f"mỗi trọng số của lớp 1 được dùng ở {rows_a[0][1]} × {rows_a[0][1]} = {vn(rows_a[0][1] ** 2, 0)} vị trí")
assert Pa == 60965224 and ungrouped == 62378344 and round(100 * pa) == 96 and round(100 * ma) == 8
assert neurons[1:] == paper_neurons[1:]  # khớp Hình 2 của bài báo, trừ lớp đầu (xem dưới)

Ba lớp dense giữ 96% tham số nhưng chỉ làm khoảng 8% phép tính. Một trọng số dense chỉ được dùng một lần cho mỗi ảnh;
một trọng số tích chập được dùng lại ở mọi vị trí của bản đồ (55 × 55 = 3.025 lần ở lớp 1). Vì vậy tham số và phép tính
là hai thước đo khác nhau của “cỡ mạng”: tham số quyết định bộ nhớ để lưu mạng, phép tính quyết định thời gian chạy.

Tổng 60.965.224 là bản chia 2 GPU, khớp “60 million parameters” của bài báo. Số nơ-ron ở lớp 2–5 khớp chuỗi
“253,440–186,624–64,896–64,896–43,264” trong chú thích Hình 2; riêng lớp đầu, 55 × 55 × 96 = 290.400 chứ không phải
253.440, một chỗ không nhất quán trong bài báo.

AlexNet thắng ILSVRC‑2012 với lỗi top‑5 15,3%, so với 26,2% của đội thứ hai. Con số 15,3% là của một tổ hợp 7 mạng CNN
(5 mạng như trên và 2 mạng có thêm một lớp tích chập, huấn luyện trước trên ImageNet Fall 2011); một mạng đơn có
18,2% trên tập kiểm định (Bảng 2 của bài báo). Ngoài GPU, bài báo nhấn mạnh ReLU (một mạng 4 lớp với ReLU đạt lỗi
huấn luyện 25% trên CIFAR‑10 nhanh gấp 6 lần tanh) và dropout ở hai lớp dense đầu.

### VGG‑16 (Simonyan & Zisserman, 2014)

Chỉ hai loại lớp: tích chập 3×3 bước 1 đệm 1 (giữ nguyên cỡ) và max pooling 2×2 bước 2. Năm khối 2‑2‑3‑3‑3 lớp tích chập
với 64‑128‑256‑512‑512 kênh, rồi dense 4096 → 4096 → 1000: tổng 13 + 3 = 16 lớp có trọng số.

In [ ]:
VGG16 = []
for n_conv, width in zip((2, 2, 3, 3, 3), (64, 128, 256, 512, 512)):
    VGG16 += [conv(3, 1, 1, width)] * n_conv + [pool()]
VGG16 += [fc(4096), fc(4096), fc(1000)]
rows_v = count(224, 3, VGG16, show=True)
Pv, Mv, pv, mv = totals(rows_v)
print(f"VGG‑16: {vn(Pv, 0)} tham số, {vn(Mv / 1e9, 2)} tỷ MAC · dense: {pct(pv, 1)} tham số, {pct(mv, 1)} MAC")
same_cost = [r[4] for r in rows_v if r[0] == "conv" and r[1] >= 28]
print(f"MAC của các lớp 3×3 trong khối 1–4: {' · '.join(vn(m / 1e9, 2) for m in same_cost)} (tỷ)")
first_fc = next(r for r in rows_v if r[0] == "fc")
print(f"riêng lớp dense đầu (7×7×512 → 4096): {vn(first_fc[3], 0)} tham số = {pct(first_fc[3] / Pv, 1)} cả mạng")
# torchvision: vgg16 num_params 138357544, GFLOPS 15,47 (con số này thực ra là số MAC); bài báo, Bảng 2: 138 triệu
assert Pv == 138357544 and round(Mv / 1e9, 2) == 15.47
assert len(set(same_cost[1:2] + same_cost[3:4] + same_cost[5:7] + same_cost[8:10])) == 1

138.357.544 tham số: đúng con số của bản VGG‑16 trong torchvision, và “138” triệu ở Bảng 2 của bài báo. Một ảnh
224×224 tốn 15,47 tỷ MAC (torchvision ghi “15,47 GFLOPS”, thực chất là đếm MAC). Riêng lớp dense đầu tiên giữ 74,3%
tham số của cả mạng, trong khi các lớp dense chỉ làm 0,8% phép tính.

Bảng từng lớp cho thấy mẫu chung của CNN kinh điển: **không gian nhỏ dần, số kênh tăng dần**. Mỗi lần pooling, số vị
trí giảm 4 lần còn số kênh gấp đôi (trọng số mỗi vị trí tăng 4 lần), nên mọi lớp 3×3 có cùng số kênh vào và ra trong
khối 1–4 đều tốn đúng 1,85 tỷ MAC. Lớp đầu của khối 2–4 (số kênh vào bằng một nửa) tốn một nửa, 0,92 tỷ; khối 5 giữ 512
kênh nên rẻ hơn.

VGG đứng thứ 2 ở phần phân loại của ILSVRC‑2014 với lỗi top‑5 7,3% (tổ hợp 7 mạng); sau cuộc thi, tổ hợp 2 mạng tốt
nhất xuống 6,8% (bài báo, mục 4.5). Bài báo thử các mạng từ 11 đến 19 lớp có trọng số: lỗi giảm dần theo độ sâu và
“bão hòa” ở 19 lớp.

Hình dưới tóm tắt phần đếm: với mỗi mạng, một thanh cho tham số và một thanh cho phép nhân–cộng, chia thành phần của
các lớp tích chập (xanh, nền đặc) và các lớp dense (cam, gạch chéo), có ghi phần trăm của dense.

In [ ]:
NETS = {
    "LeNet (MNIST, mục 6)": count(28, 1, MNIST_NETS["max"]),
    "LeNet‑5 (1998)": rows5,
    "AlexNet (2012)": rows_a,
    "VGG‑16 (2014)": rows_v,
}
fig, ax = plt.subplots(figsize=(9, 3.8))
labels = []
for i, (name, rows) in enumerate(NETS.items()):
    _, _, p_d, m_d = totals(rows)
    for jj, (what, share) in enumerate([("tham số", p_d), ("phép nhân–cộng", m_d)]):
        y = 2.6 * i + jj
        ax.barh(y, 100 * (1 - share), color=BLUE, height=0.8)
        ax.barh(y, 100 * share, left=100 * (1 - share), color=ORANGE, height=0.8, hatch="//", edgecolor="white", lw=0)
        ax.text(101, y, f"dense {pct(share, 1)}", va="center", fontsize=9)
        labels.append((y, f"{name} · {what}"))
ax.set_yticks([y for y, _ in labels], [t for _, t in labels], fontsize=9)
ax.invert_yaxis()
ax.set(xlim=(0, 125), xlabel="phần trăm của cả mạng (xanh: tích chập · cam gạch chéo: dense)")
ax.set_xticks([0, 25, 50, 75, 100])
vn_axes(fig, "x")
plt.tight_layout()
plt.show()

Ở mọi mạng có lớp dense lớn, tham số dồn về dense còn phép tính dồn về tích chập. LeNet‑5 là ngoại lệ về tham số chỉ vì
bài báo gọi C5 là một lớp tích chập (5×5 trên bản đồ 5×5, nên thực chất là nối đầy đủ); ta theo cách gọi đó.

## 9 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

**Sửa kiến trúc rồi đếm lại.** Ba thay đổi: AlexNet với đúng ảnh 224×224 như bài báo viết; VGG‑16 thay ba lớp dense
bằng pooling trung bình toàn cục (lấy trung bình mỗi kênh trên cả bản đồ 7×7, như ResNet ở Bài 13) rồi một lớp dense
512 → 1000; và VGG‑16 bỏ lần pooling cuối.

In [ ]:
experiments = {
    "AlexNet, ảnh 224×224": (224, ALEXNET),
    "VGG‑16, GAP + dense 1000": (224, VGG16[:-4] + [pool(), GAP, fc(1000)]),
    "VGG‑16, bỏ pooling cuối": (224, VGG16[:-4] + VGG16[-3:]),
}
for name, (n, layers) in experiments.items():
    rows = count(n, 3, layers)
    P_, M_, p_d, _ = totals(rows)
    k_fc = next(i for i, r in enumerate(rows) if r[0] == "fc")
    _, n_last, c_last, _, _ = rows[k_fc - 1]
    print(
        f"{name:<26}: {vn(P_, 0):>12} tham số, {vn(M_ / 1e9, 2):>5} tỷ MAC · dense {pct(p_d, 1)} tham số"
        f" · lớp dense đầu nhận {n_last}×{n_last}×{c_last} = {vn(n_last**2 * c_last, 0)} giá trị,"
        f" có {vn(rows[k_fc][3], 0)} tham số"
    )

Ba kết quả đáng chú ý:
- **AlexNet với ảnh 224:** lớp 1 ra 54×54 chứ không phải 55×55. Các lớp sau nhỏ dần theo, và lớp dense đầu chỉ nhận
  5×5×256 = 6.400 giá trị thay vì 6×6×256 = 9.216. Trọng số của bài báo không còn khớp cỡ mạng.
- **VGG‑16 với GAP:** bỏ ba lớp dense, số tham số giảm từ 138 triệu xuống khoảng 15 triệu, còn số phép tính gần như không
  đổi, vì phép tính vốn nằm ở các lớp tích chập. Các mạng sau VGG (GoogLeNet, ResNet) đều kết thúc bằng pooling trung
  bình toàn cục.
- **VGG‑16 bỏ pooling cuối:** bản đồ trước lớp dense còn 14×14, nên riêng lớp dense đầu đã có hơn 400 triệu tham số.

**Đổi bộ lọc làm mờ.** Zhang (2019) thử nhiều bộ lọc trước khi lấy mẫu thưa: $[1\ 1]$ (tương đương average pooling),
$[1\ 2\ 1]$ (dùng ở trên) và $[1\ 4\ 6\ 4\ 1]$. Ô dưới đo lại độ thay đổi khi dời 1 điểm ảnh như ở mục 3, với từng bộ lọc.

In [ ]:
def maxblur(x, taps):
    """MaxBlurPool với bộ lọc tùy chọn: max 2×2 bước 1, làm mờ taps ⊗ taps (chuẩn hóa), rồi lấy bước 2."""
    H, W = x.shape[1:3]
    xp = np.pad(x, [(0, 0), (0, 1), (0, 1)], mode="edge")
    m = np.maximum(np.maximum(xp[:, :H, :W], xp[:, 1:, :W]), np.maximum(xp[:, :H, 1:], xp[:, 1:, 1:]))
    w = np.array(taps, float) / sum(taps)
    lo_pad, hi_pad = (len(w) - 1) // 2, len(w) // 2
    mp = np.pad(m, [(0, 0), (lo_pad, hi_pad), (lo_pad, hi_pad)], mode="reflect")
    out = sum(w[i] * w[j] * mp[:, i : i + H, j : j + W] for i in range(len(w)) for j in range(len(w)))
    return out[:, ::2, ::2]


blur_change = {}
for taps in [(1,), (1, 1), (1, 2, 1), (1, 4, 6, 4, 1)]:
    base = maxblur(X2, taps)
    blur_change[taps] = rel_change(maxblur(shift(X2, 1, 0), taps), base)
    print(f"bộ lọc {vec(taps):<14}: dời 1 điểm ảnh → đổi {pct(blur_change[taps], 1)}")
assert abs(blur_change[(1,)] - sens["max pooling"][0]) < 1e-9  # bộ lọc [1] = max pooling thường
assert abs(blur_change[(1, 2, 1)] - sens["MaxBlurPool"][0]) < 1e-9
assert blur_change[(1, 4, 6, 4, 1)] < blur_change[(1, 2, 1)] < blur_change[(1, 1)] < blur_change[(1,)]

Bộ lọc $[1]$ (không làm mờ) cho đúng max pooling thường, $[1\ 2\ 1]$ cho đúng MaxBlurPool ở mục 3. Bộ lọc càng rộng thì
đầu ra càng ít đổi khi dời, nhưng cũng càng mờ: mạng mất bớt chi tiết. Zhang (2019) thấy trên ImageNet độ chính xác vẫn
tăng nhẹ với các bộ lọc này, nhưng mạng vẫn chưa ổn định hoàn toàn với phép dời.

**Huấn luyện lại với cách pooling khác.** `train_lenet()` nhận `kind="avg"` cho average pooling, và `seed` khác. Ô
dưới huấn luyện average pooling 200 bước và so với max pooling sau cùng số bước, trên 2.000 ảnh kiểm tra đầu để chạy
nhanh. Tăng `steps` (600 như mục 6, hoặc cả lượt: 938 bước) nếu bạn có thêm thời gian.

In [ ]:
kind, seed, steps, n_eval = "avg", 0, 200, 2000
for k_ in (kind, "max"):
    P_ = train_lenet(k_, seed=seed, steps=steps)
    p0 = predict(P_, Xte[:n_eval], k_)
    ch = np.mean([(predict(P_, shift(Xte[:n_eval], *d), k_) != p0).mean() for d in DIRS])
    print(
        f"{k_}, seed {seed}, {steps} bước, {vn(n_eval, 0)} ảnh kiểm tra: đúng {pct((p0 == yte[:n_eval]).mean())}"
        f" · dời 1 điểm ảnh đổi {pct(ch)}"
    )

Sau 200 bước cả hai mạng vẫn đang học, nên độ chính xác còn thấp hơn mục 6. Một seed và một lần chạy ngắn chưa nói được
cách pooling nào ổn định hơn; muốn so thì xem khoảng 3 seed ở mục 7, hoặc chạy ô này với vài `seed`.

## 10 · VGG: sâu hơn, rồi chạm bức tường độ sâu

VGG cho thấy 16–19 lớp tốt hơn 11 lớp. Vậy cứ thêm lớp là tốt hơn? He et al. (2015) thử trên CIFAR‑10 với các mạng
**trơn** (plain: chỉ xếp chồng các lớp, không có kết nối tắt) gồm $6n + 2$ lớp có trọng số: một lớp tích chập đầu, ba
chặng mỗi chặng $2n$ lớp 3×3, rồi pooling trung bình toàn cục (GAP) và một lớp dense 10. Mọi lớp tích chập là
Conv → BatchNorm → ReLU.

Script `depth.py` của demo làm lại thí nghiệm đó, thu nhỏ cho MNIST: lớp đầu là tích chập 3×3 bước 2 (28 → 14), ba
chặng có 8, 16, 32 kênh, lớp đầu của chặng 2 và 3 có bước 2 (14 → 7 → 4). Độ sâu 8, 20, 32, 44, 56 lớp; SGD lô 64,
3 lượt trên 55.000 ảnh, $\eta \in \{0{,}03;\ 0{,}1;\ 0{,}3\}$ × 3 seed. $\eta$ của mỗi độ sâu được chọn trên 5.000 ảnh
kiểm định (cuối tệp huấn luyện); lỗi huấn luyện đo trên 10.000 ảnh huấn luyện đầu, sau khi học xong.

Trước khi đọc kết quả, `count()` của mục 8 tính lại số tham số của từng mạng; chúng phải khớp các mạng đã huấn luyện.

In [ ]:
def plain_net(depth, widths=(8, 16, 32)):
    """Mạng trơn kiểu He et al. (2015) cho MNIST, như depth.py: 6n + 2 lớp có trọng số."""
    per_stage = (depth - 2) // 3  # 2n lớp 3×3 mỗi chặng
    layers = [conv(3, 2, 1, widths[0]), BN]
    for stage, w in enumerate(widths):
        for i in range(per_stage):
            layers += [conv(3, 2 if stage > 0 and i == 0 else 1, 1, w), BN]
    return layers + [GAP, fc(10)]


DEPTHS, ETAS = depth_runs["depths"], depth_runs["etas"]
cfg = {(r["depth"], r["eta"]): r["seeds"] for r in depth_runs["runs"]}
print(
    f"{len(DEPTHS)} độ sâu × {len(ETAS)} giá trị η × {len(depth_runs['seeds'])} seed, {depth_runs['epochs']} lượt "
    f"trên {vn(depth_runs['nTrain'], 0)} ảnh; lỗi huấn luyện đo trên {vn(depth_runs['nTrainEval'], 0)} ảnh huấn luyện đầu"
)
for d in DEPTHS:
    rows_d = count(28, 1, plain_net(d))
    sizes = sorted({r[1] for r in rows_d if r[0] == "conv"}, reverse=True)
    n_layers = sum(r[0] in ("conv", "fc") for r in rows_d)
    P_d = totals(rows_d)[0]
    print(f"{d:>2} lớp: {n_layers} lớp có trọng số, bản đồ {' → '.join(map(str, sizes))}, {vn(P_d, 0):>7} tham số")
    assert n_layers == d and P_d == cfg[(d, ETAS[0])][0]["params"], "khác với các mạng mà demo đã huấn luyện"

### 🤔 Dự đoán trước
Mạng 56 lớp có hơn gấp ba số tham số của mạng 20 lớp, và chứa nó: đặt 36 lớp thêm vào làm hàm đồng nhất là ra đúng
mạng 20 lớp. Sau cùng 3 lượt, mỗi mạng với $\eta$ tốt nhất của nó, lỗi trên chính **dữ liệu huấn luyện** của mạng
56 lớp sẽ thấp hơn, bằng, hay cao hơn mạng 20 lớp?

In [ ]:
def val_median(d, e):
    return np.median([s.get("val", 1.0) for s in cfg[(d, e)]])  # lần chạy nổ (NaN) tính lỗi 100%, như demo


best_eta = {d: min(ETAS, key=lambda e: val_median(d, e)) for d in DEPTHS}
err = {d: {k: [s[k] for s in cfg[(d, best_eta[d])]] for k in ("train", "val", "test")} for d in DEPTHS}
for d in DEPTHS:
    e = err[d]
    print(
        f"{d:>2} lớp, η = {vn(best_eta[d], 2)}: lỗi huấn luyện {spread(e['train'], d=2):<13} · kiểm định "
        f"{spread(e['val'], d=2):<13} · kiểm tra {spread(e['test'], d=2)}"
    )
best_d = min(DEPTHS, key=lambda d: np.median(err[d]["train"]))
print(
    f"lỗi huấn luyện thấp nhất: {best_d} lớp; 56 lớp so với {best_d} lớp: "
    f"{verdict(err[56]['train'], err[best_d]['train'], False)}"
)

Mạng 56 lớp có lỗi trên **chính dữ liệu huấn luyện** cao hơn rõ so với mạng 20 lớp: khoảng 3 seed của hai mạng không
chồng nhau. Lỗi giảm từ 8 lên 20 lớp, rồi tăng đều từ 20 đến 56 lớp. Hình dưới vẽ bảng trên: mỗi độ sâu có lỗi huấn
luyện (chấm tròn xanh, nét liền) và lỗi kiểm định (ô vuông cam, nét gạch); chấm nhỏ là từng seed, đường nối trung vị.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for key, label, col, mk, ls, dx in [
    ("train", "lỗi huấn luyện (10.000 ảnh huấn luyện)", BLUE, "o", "-", -0.6),
    ("val", "lỗi kiểm định (5.000 ảnh)", ORANGE, "s", "--", 0.6),
]:
    med = [100 * np.median(err[d][key]) for d in DEPTHS]
    ax.plot(DEPTHS, med, color=col, ls=ls, marker=mk, ms=8, lw=2, label=label)
    for d in DEPTHS:
        ax.plot([d + dx] * 3, 100 * np.array(err[d][key]), mk, color=col, ms=4, mfc="white")
ax.set(xlabel="số lớp có trọng số", ylabel="lỗi sau 3 lượt (%)", xticks=DEPTHS, ylim=(0, None))
ax.set_title("Mạng trơn trên MNIST: từ 20 lớp trở lên, thêm lớp thì lỗi huấn luyện tăng (3 seed)", fontsize=10)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, fontsize=9)
vn_axes(fig)
plt.tight_layout()
plt.show()

Đây không phải **quá khớp** (Bài 08): quá khớp là lỗi huấn luyện thấp mà lỗi kiểm định cao. Ở đây lỗi kiểm định đi
sát lỗi huấn luyện, và mạng sâu tệ hơn ngay trên những ảnh nó đã học. He et al. gọi đây là **vấn đề suy thoái**
(degradation): “adding more layers to a suitably deep model leads to higher training error”.

Có phải do $\eta$? Ô dưới so 56 lớp với 20 lớp ở **từng** giá trị $\eta$, không chọn gì cả.

In [ ]:
for e in ETAS:
    a, b = [s["train"] for s in cfg[(56, e)]], [s["train"] for s in cfg[(20, e)]]
    print(f"η = {vn(e, 2)}: 56 lớp {spread(a, d=2):<13} · 20 lớp {spread(b, d=2):<13} → 56 lớp {verdict(a, b, False)}")

Ở cả ba tốc độ học, mạng 56 lớp kém hơn rõ. Hình dưới vẽ lỗi huấn luyện *trong lúc* học (trung bình của mỗi 100 lô,
trung vị của 3 seed, trục dọc thang log). Mạng 20 lớp (xanh, nét liền) và 56 lớp (cam, nét gạch) được tô màu; các độ
sâu khác là đường xám mảnh, có ghi số lớp ở cuối đường.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
steps_axis = None
for d in DEPTHS:
    curves = np.array([s["curve"] for s in cfg[(d, best_eta[d])]])  # (3 seed, số điểm, [mất mát, lỗi])
    med = np.median(curves[:, :, 1], axis=0)
    steps_axis = depth_runs["log"] * np.arange(1, len(med) + 1)
    style = {20: (BLUE, "-", 2.2), 56: (ORANGE, "--", 2.2)}.get(d, (GREY, "-", 1))
    ax.plot(steps_axis, 100 * med, color=style[0], ls=style[1], lw=style[2])
    ax.annotate(
        f"{d} lớp",
        (steps_axis[-1], 100 * med[-1]),
        xytext=(4, 0),
        textcoords="offset points",
        fontsize=8,
        va="center",
        color="black" if d in (20, 56) else GREY,
    )
ax.set(yscale="log", xlabel="bước SGD", ylabel="lỗi huấn luyện đang chạy (%)", xlim=(0, steps_axis[-1] * 1.12))
ax.set_title("Lỗi huấn luyện trong 3 lượt (trung vị 3 seed, η tốt nhất mỗi độ sâu)", fontsize=10)
ax.grid(alpha=0.3, which="both")
vn_axes(fig)
plt.tight_layout()
plt.show()

Mạng 56 lớp kém hơn trong **suốt** quá trình học, không chỉ ở cuối, như Hình 1 của He et al. Có thể nó chỉ cần học lâu
hơn? Demo cũng huấn luyện 20 và 56 lớp trong 10 lượt (chỉ seed 0, $\eta$ 0,03 và 0,1):

In [ ]:
long = depth_runs["long"]
best_long = {d: min((r for r in long if r["depth"] == d), key=lambda r: r["val"]) for d in (20, 56)}
for d, r in best_long.items():
    print(
        f"{depth_runs['longEpochs']} lượt, {d} lớp (η = {vn(r['eta'], 2)}, chọn trên kiểm định): lỗi huấn luyện "
        f"{pct(r['train'])} · kiểm định {pct(r['val'])} · kiểm tra {pct(r['test'])}"
    )
print(f"3 lượt, 20 lớp (trung vị 3 seed): lỗi huấn luyện {pct(np.median(err[20]['train']))}")

Sau 10 lượt, mạng 56 lớp vẫn còn 3,06% lỗi huấn luyện, cao hơn cả mạng 20 lớp chỉ sau 3 lượt. Đây chỉ là một seed nên
là kiểm tra thêm chứ không phải kết luận, nhưng nó cho thấy khoảng cách không chỉ vì học chưa đủ lâu.

Vì sao mạng sâu hơn lại học kém hơn? Không phải vì nó **không thể** làm tốt: lấy mạng 20 lớp đã học, chèn 36 lớp làm
hàm đồng nhất, ta được một mạng 56 lớp có đúng lỗi của mạng 20 lớp. He et al. viết: “There exists a solution by
construction to the deeper model”. Cũng không đơn thuần là gradient biến mất (Bài 09): mọi lớp đã có BatchNorm, và
He et al. kiểm tra rằng gradient ở các mạng trơn của họ có độ lớn bình thường. Vấn đề là **tối ưu**: SGD không tìm ra lời
giải tốt mà mạng sâu chắc chắn có, vì một chồng lớp phi tuyến rất khó học đúng hàm đồng nhất.

Bài 13 sửa điều đó bằng một thay đổi rất nhỏ: cho mỗi cặp lớp tính $x + F(x)$ thay vì $F(x)$. Khi đó hàm đồng nhất chỉ
cần $F = 0$, điều rất dễ học, và thêm lớp không còn làm mạng học kém đi. Trên CIFAR‑10, He et al. báo cáo lỗi kiểm tra
8,75% cho ResNet 20 lớp và 6,97% cho ResNet 56 lớp (Bảng 6).

## Tóm tắt
- Pooling 2×2 bước 2 thu nhỏ mỗi cạnh một nửa và không có tham số. Dời ảnh 2 điểm ảnh thì đầu ra dời đúng 1 ô; dời
  1 điểm ảnh thì không lưới nào theo được, và bản đồ 14×14 của max pooling đổi 46,5%.
- $n_{\text{out}} = \lfloor (n_{\text{in}} + 2p - k)/s \rfloor + 1$, kiểm tra trên 3.644 tổ hợp; phần lẻ bị bỏ, nên
  AlexNet cần ảnh 227 chứ không phải 224.
- Vùng tiếp nhận: $r \leftarrow r + (k-1)j$, $j \leftarrow js$. LeNet nhìn 16×16, VGG‑16 nhìn 212×212 chỉ bằng
  bộ lọc 3×3; hai lớp 3×3 nhìn như một lớp 5×5 với $18C^2$ thay vì $25C^2$ trọng số.
- Max pooling không làm mạng bất biến: LeNet dời 1 điểm ảnh đổi 1,3–3,9% dự đoán (3 seed), ngang mạng không pooling;
  dời 4 điểm ảnh thì pooling giúp rõ (50,3–58,6% so với 47,1–48,8%), nhưng mạng tốt nhất cũng chỉ còn 56,7–63,3%.
- Đếm từng lớp khớp đúng số đã công bố: LeNet‑5 60.000, AlexNet 60.965.224, VGG‑16 138.357.544 tham số. Tham số dồn ở
  dense (AlexNet 96%), phép tính dồn ở tích chập (dense chỉ 8%).
- Mạng trơn sâu hơn học kém hơn: 56 lớp có lỗi huấn luyện 5,78–12,84%, 20 lớp 0,87–1,10%. Không phải quá khớp, mà là
  khó tối ưu (vấn đề suy thoái).

## Câu hỏi ôn tập
1. Vì sao dời ảnh 2 điểm ảnh thì đầu ra của max pooling 2×2 dời đúng 1 ô, còn dời 1 điểm ảnh thì đầu ra đổi hẳn?
2. Một lớp tích chập 7×7 bước 2 đệm 3 nhận bản đồ 224×224. Đầu ra cỡ bao nhiêu? Bao nhiêu hàng bị bỏ?
3. Hai lớp 3×3 bước 1, rồi max pooling 2×2 bước 2, rồi một lớp 3×3 bước 1: vùng tiếp nhận là bao nhiêu?
4. Ở AlexNet, vì sao ba lớp dense giữ 96% tham số mà chỉ làm khoảng 8% phép tính?
5. LeNet có max pooling đổi 1,3–3,9% dự đoán khi dời 1 điểm ảnh, LeNet không pooling đổi 1,8–2,2%. Có kết luận được
   mạng nào ổn định hơn không?
6. Mạng trơn 56 lớp có lỗi huấn luyện cao hơn mạng 20 lớp. Vì sao đó không phải quá khớp, và vì sao đáng ngạc nhiên?

<details><summary>Gợi ý đáp án</summary>

1. Max pooling = lấy max trên mọi cửa sổ (dời theo đầu vào) rồi giữ một ô trong hai. Dời 2 điểm ảnh thì các ô được giữ
   vẫn là cùng các cửa sổ, chỉ dời đi 1 ô; dời 1 điểm ảnh thì các ô được giữ là những cửa sổ khác hẳn.
2. $\lfloor (224 + 6 - 7)/2 \rfloor + 1 = \lfloor 111{,}5 \rfloor + 1 = 112$; $223 \bmod 2 = 1$ hàng bị bỏ (lớp đầu của
   ResNet).
3. $r = 1 + 2 + 2 = 5$, $j = 1$; pooling: $r = 5 + 1 = 6$, $j = 2$; lớp cuối: $r = 6 + 2 \cdot 2 = 10$. Vùng 10×10.
4. Một trọng số dense chỉ được dùng một lần cho mỗi ảnh; một trọng số tích chập được dùng ở mọi vị trí của bản đồ
   (3.025 vị trí ở lớp 1), nên ít tham số mà nhiều phép tính.
5. Không: hai khoảng 3 seed chồng lên nhau, nên khác biệt nằm trong dao động giữa các seed.
6. Quá khớp là lỗi huấn luyện thấp mà lỗi kiểm định cao; ở đây mạng sâu tệ hơn ngay trên dữ liệu huấn luyện. Đáng ngạc
   nhiên vì mạng 56 lớp chứa mạng 20 lớp (các lớp thêm là hàm đồng nhất), nên nó ít nhất phải làm được như vậy; SGD
   không tìm ra lời giải đó.
</details>

## Tìm hiểu thêm
- Y. LeCun, L. Bottou, Y. Bengio & P. Haffner (1998), [Gradient-Based Learning Applied to Document
  Recognition](https://doi.org/10.1109/5.726791): LeNet‑5, mục II‑B có số tham số từng lớp.
- A. Krizhevsky, I. Sutskever & G. Hinton (2012), [ImageNet Classification with Deep Convolutional Neural
  Networks](https://papers.nips.cc/paper_files/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html):
  AlexNet; Bảng 2 có lỗi của 1, 5 và 7 mạng.
- K. Simonyan & A. Zisserman (2014), [Very Deep Convolutional Networks for Large-Scale Image
  Recognition](https://arxiv.org/abs/1409.1556): VGG; mục 2.3 về chồng các lớp 3×3.
- K. He, X. Zhang, S. Ren & J. Sun (2015), [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385):
  Hình 1 và mục 4.1 về vấn đề suy thoái.
- A. Araujo, W. Norris & J. Sim (2019), [Computing Receptive Fields of Convolutional Neural
  Networks](https://distill.pub/2019/computing-receptive-fields/), Distill: công thức vùng tiếp nhận và bảng các mạng.
- W. Luo, Y. Li, R. Urtasun & R. Zemel (2016), [Understanding the Effective Receptive Field in Deep Convolutional
  Neural Networks](https://arxiv.org/abs/1701.04128).
- R. Zhang (2019), [Making Convolutional Networks Shift-Invariant Again](https://arxiv.org/abs/1904.11486): MaxBlurPool.
- A. Azulay & Y. Weiss (2019), [Why do deep convolutional networks generalize so poorly to small image
  transformations?](https://arxiv.org/abs/1805.12177)
- CS231n, [Convolutional Neural Networks](https://cs231n.github.io/convolutional-networks/): công thức kích thước, bảng
  bộ nhớ và tham số của VGG‑16, ghi chú về ảnh 227 của AlexNet.

**Bài tiếp theo:** [Bài 13 · Kết nối tắt & ResNet](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/resnet/): thêm $x$ vào
đầu ra của mỗi cặp lớp, và mạng sâu không còn học kém hơn mạng nông.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")